In [ ]:
from label_studio_sdk import Client
import json

# Initialize the Label Studio client
label_studio_url = 'http://35.208.227.251'
api_token = '11c87e17e3b38e044997fbb770e31515294aa328'
client = Client(url=label_studio_url, api_key=api_token)

# Get your project
project_id = 1  # Replace with your project ID
project = client.get_project(project_id)

# Get a specific task
task_id = 7  # Replace with your task ID
task = project.get_task(task_id)

# Get annotations for the task
annotations = task

# Save to JSON file
output_file = f"task_{task_id}_annotations.json"
with open(output_file, 'w') as f:
    json.dump(annotations, f, indent=4)

print(f"Saved {len(annotations)} annotations to {output_file}")

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

# Configuration
IMAGE_PATH = 'your_image.jpg'  # Path to your downloaded image
ANNOTATION_FILE = 'task_7_annotations.json'  # Your annotation file
LABEL_COLORS = {
    'male duck': 'blue',
    'female duck': 'red',
    'Juvenile duck': 'yellow',
    'Ice': 'green',
    'duck': 'orange'
}

def plot_keypoints():
    # Load image
    img = cv2.imread(IMAGE_PATH)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB for matplotlib
    
    # Load annotations
    with open(ANNOTATION_FILE) as f:
        task_data = json.load(f)
    
    # Create figure
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    
    # Plot each annotation
    for annotation in task_data.get('annotations', []):
        for result in annotation.get('result', []):
            if result['type'] == 'keypointlabels':
                value = result['value']
                
                # Get coordinates (assuming percentage coordinates)
                x = (value['x'] / 100) * img.shape[1]
                y = (value['y'] / 100) * img.shape[0]
                
                # Get label and color
                label = value['labels'][0]
                color = LABEL_COLORS.get(label, 'gray')  # Default to gray for unknown labels
                
                # Plot the point
                plt.scatter(x, y, color=color, s=100, edgecolors='white', label=label)
    
    # Create legend with unique labels
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))  # Remove duplicates
    plt.legend(by_label.values(), by_label.keys(), title='Keypoints')
    
    plt.title('Image with Keypoint Annotations')
    plt.axis('off')
    plt.show()

if __name__ == '__main__':
    plot_keypoints()

In [ ]:
from label_studio_sdk import Client
import json

# SOURCE INSTANCE CONFIG
SOURCE_LS_URL = 'http://35.208.227.251'
SOURCE_API_TOKEN = '11c87e17e3b38e044997fbb770e31515294aa328'
SOURCE_PROJECT_ID = 1  # Source project ID
SOURCE_TASK_ID = 7     # Source task ID

# TARGET INSTANCE CONFIG
TARGET_LS_URL = 'http://localhost:8081/'
TARGET_API_TOKEN = ''
TARGET_PROJECT_ID = 181  # Target project ID
TARGET_TASK_ID = 70986  # Target task ID

def transfer_annotations():
    # Connect to source instance
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    source_project = source_client.get_project(SOURCE_PROJECT_ID)
    source_task = source_project.get_task(SOURCE_TASK_ID)
    
    # Get source annotations
    source_annotations = source_task.annotations
    print(f"Found {len(source_annotations)} annotations in source task")

    # Connect to target instance
    target_client = Client(url=TARGET_LS_URL, api_key=TARGET_API_TOKEN)
    target_project = target_client.get_project(TARGET_PROJECT_ID)
    
    # Verify target task exists
    try:
        target_task = target_project.get_task(TARGET_TASK_ID)
    except Exception as e:
        raise ValueError(f"Target task {TARGET_TASK_ID} not found") from e

    # Prepare annotation payload
    for annotation in source_annotations:
        # Create clean annotation payload
        payload = {
            "result": annotation.get('result', []),
            "was_cancelled": annotation.get('was_cancelled', False),
            "ground_truth": annotation.get('ground_truth', False),
            # Add other fields as needed
        }

        # Post to target instance
        response = target_client.make_request(
            'POST',
            f'/api/tasks/{TARGET_TASK_ID}/annotations/',
            json=payload
        )

        if response.status_code == 201:
            print(f"Successfully transferred annotation {annotation['id']}")
        else:
            print(f"Failed to transfer annotation {annotation['id']}: {response.text}")

    print("Annotation transfer completed")

def main():
    # Optional: Save source annotations first
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    source_task = source_client.get_project(SOURCE_PROJECT_ID).get_task(SOURCE_TASK_ID)
    
    with open(f'source_annotations_{SOURCE_TASK_ID}.json', 'w') as f:
        json.dump(source_task.annotations, f, indent=2)
    
    # Perform transfer
    transfer_annotations()

if __name__ == '__main__':
    main()

In [ ]:
import json
import cv2
import numpy as np


from label_studio_sdk import Client
import json

# Initialize the Label Studio client
label_studio_url = 'http://35.208.227.251'
api_token = '11c87e17e3b38e044997fbb770e31515294aa328'
client = Client(url=label_studio_url, api_key=api_token)

# Get your project
project_id = 1  # Replace with your project ID
project = client.get_project(project_id)

# Get a specific task
task_id = 31  # Replace with your task ID
task = project.get_task(task_id)

# Get annotations for the task
annotations = task

# Save to JSON file
output_file = f"task_{task_id}_annotations.json"
with open(output_file, 'w') as f:
    json.dump(annotations, f, indent=4)

print(f"Saved {len(annotations)} annotations to {output_file}")

# Configuration
IMAGE_PATH = '/Users/aqeelpatel/Documents/Personal website/muhammed10patel.github.io/assets/img/project_6/aed014f4-IMG_4641.JPG'          # Path to your downloaded image
ANNOTATION_FILE = output_file  # Your annotation file
OUTPUT_IMAGE = '/Users/aqeelpatel/Documents/Personal website/muhammed10patel.github.io/assets/img/project_6/aed014f4-IMG_4641_annotated.png'   # Output file name

# Label colors in BGR format (OpenCV uses BGR instead of RGB)
LABEL_COLORS = {
    'Adult Male': (255, 0, 0),         # Blue
    'Adult Female': (0, 0, 255),       # Red
    'Juvenile': (0, 255, 255),   # Yellow
    'Ice': (0, 255, 0),               # Green
    'Unknown': (0, 165, 255)             # Orange
}

# Keypoint display settings
POINT_RADIUS = 4
BORDER_THICKNESS = 1
TEXT_THICKNESS = 2
TEXT_SCALE = 1.2

def plot_and_save_keypoints():
    # Load image
    img = cv2.imread(IMAGE_PATH)
    if img is None:
        print(f"Error: Could not load image from {IMAGE_PATH}")
        return

    # Load annotations
    with open(ANNOTATION_FILE) as f:
        task_data = json.load(f)

    # Draw annotations
    for annotation in task_data.get('annotations', []):
        for result in annotation.get('result', []):
            if result['type'] == 'keypointlabels':
                value = result['value']
                
                # Convert percentage coordinates to absolute pixels
                x = int((value['x'] / 100) * img.shape[1])
                y = int((value['y'] / 100) * img.shape[0])
                
                # Get label and color
                label = value['keypointlabels'][0]
                color = LABEL_COLORS.get(label, (128, 128, 128))  # Default to gray

                # Draw keypoint with border
                cv2.circle(img, (x, y), POINT_RADIUS + BORDER_THICKNESS, (255, 255, 255), -1)
                cv2.circle(img, (x, y), POINT_RADIUS, color, -1)
                
                # # Add label text
                # cv2.putText(img, label, (x + POINT_RADIUS + 5, y + 5),
                #             cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE,
                #             (255, 255, 255), TEXT_THICKNESS)

    # Add legend
    legend_x = 50
    legend_y = 50
    for i, (label, color) in enumerate(LABEL_COLORS.items()):
        y_pos = legend_y + i * 50
        # Draw color swatch
        cv2.rectangle(img, (legend_x, y_pos), (legend_x + 30, y_pos + 30), color, -1)
        # Add label text
        cv2.putText(img, label, (legend_x + 40, y_pos + 25), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    # Save high-quality image
    cv2.imwrite(OUTPUT_IMAGE, img, [cv2.IMWRITE_JPEG_QUALITY, 100])
    print(f"Saved annotated image to {OUTPUT_IMAGE} with high resolution")

    # Optional: Display the image (press any key to close)
    # cv2.imshow('Annotated Image', img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

if __name__ == '__main__':
    plot_and_save_keypoints()